In [ ]:
'''
Kerangka:
Terdapat beberapa hal yang harus diperhatikan:
1. Jumlah kelas tidak seimbang 
- Jumlah kelas pada data tidak seimbang, jadi harus menggunakan stratified split
2. Memastikan bahwa hasil augmented (ada variasi brightness, dirotasi, dan di-flip secara horizontal) dari setiap data tersebar merata ke train, val, dan test set. jadi, hasil preprocessing tidak mendominasi hanya salah satu training set/val/test.  
- Ide: setiap data original yang masuk ke train/val/test, dia harus "ngebawa" versi preprocessed (LENGKAP TIGA-TIGANYA) dari dirinya sendiri. langkah-langkah:
- Steps:
1. Pada dataframe /home/g6422078/Data Preprocessing/Training Set/NA x BA/df_preprocessed_NA_BA.csv, bagi menjadi dua kelompok. kelompok pertama menyimpan gambar yang hanya di-preprocessed dengan resizing dan rescaling. kelompok kedua menyimpan gambar yang di-preprocessed dan augmented. kedua kelompok dapat dibedakan berdasarkan image_path nya
- kelompok 1: memiliki image_path /home/g6422078//Data Preprocessing/Training Set/NA x BA/Nonaugmented/<nama_image>
- kelompok 2: memiliki image_path 
   - /home/g6422078//Data Preprocessing/Training Set/NA x BA/Brightness Varied/<nama_image>, atau
   - /home/g6422078//Data Preprocessing/Training Set/NA x BA/Horizontal Flip/<nama_image>, atau
   - /home/g6422078//Data Preprocessing/Training Set/NA x BA/Rotated/<nama_image>
2. Bagi dataframe dataframe yang image path nya hanya memuat image path ke gambar yang sudah di-preprocessed HANYA resizing dan rescaling ke train, test, split set menggunakan stratified split 
2. Tambahkan versi augmented dari dirinya ke train/test/split set (sesuaikan. jika image_123 masuk sebagai train set maka versi rotated, varied in brightness, dan horizontally flipped juga masuk sebagai train set)
3. Simpan masing-massing set ke dalam dataframe (df_train, df_val, dan df_test)

Struktur dataframe /home/g6422078/Data Preprocessing/Training Set/NA x BA/df_preprocessed_NA_BA.csv adalah:
image_path	Cluster
0	/home/g6422078/Data Preprocessing/Training Set...	1
1	/home/g6422078/Data Preprocessing/Training Set...	4
2	/home/g6422078/Data Preprocessing/Training Set...	2
3	/home/g6422078/Data Preprocessing/Training Set...	4
4	/home/g6422078/Data Preprocessing/Training Set...	4
'''

In [2]:
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd

In [3]:
df = pd.read_csv(r"D:\UNI LIFE\TUGAS AKHIR\Data Preprocessing\New Ver. (Preprocessing before splitting)\df_preprocessed_NA_BA (revisi pada nilai cluster data).csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8968 entries, 0 to 8967
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  8968 non-null   object
 1   Cluster     8968 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 140.3+ KB


In [5]:
df.value_counts('Cluster')

Cluster
4    2140
1    1992
3    1752
2    1736
0    1348
Name: count, dtype: int64

In [6]:
# pisahkan augmented vs nonaugmented
df_nonaug = df[df["image_path"].str.contains("Nonaugmented")]
df_aug = df[~df["image_path"].str.contains("Nonaugmented")]

In [7]:
# generate "ID gambar asli"
'''
Kita butuh cara buat tahu bahwa semua versi dari gambar ini berasal dari gambar yang sama
'''

import os

def get_image_id(path):
    filename = os.path.basename(path)
    return filename  # asumsi nama sama untuk semua versi

df_nonaug["image_id"] = df_nonaug["image_path"].apply(get_image_id)
df_aug["image_id"] = df_aug["image_path"].apply(get_image_id)

C:\Users\USER\AppData\Local\Temp\ipykernel_14364\2462834037.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nonaug["image_id"] = df_nonaug["image_path"].apply(get_image_id)
C:\Users\USER\AppData\Local\Temp\ipykernel_14364\2462834037.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aug["image_id"] = df_aug["image_path"].apply(get_image_id)


In [8]:
df_nonaug

,image_path,Cluster,image_id
0,/home/g6422078/Data Preprocessing/Training Set...,1,Tanggulangi-Pemanasan-Global-Korea-Bantu-Rp30-...
1,/home/g6422078/Data Preprocessing/Training Set...,1,IMG_832.jpg
2,/home/g6422078/Data Preprocessing/Training Set...,2,Screenshot 2026-01-15 051301.png
3,/home/g6422078/Data Preprocessing/Training Set...,4,IMG_251.jpg
4,/home/g6422078/Data Preprocessing/Training Set...,4,IMG_46.jpg
...,...,...,...
2237,/home/g6422078/Data Preprocessing/Training Set...,4,IMG_880.jpg
2238,/home/g6422078/Data Preprocessing/Training Set...,4,Screenshot 2026-01-07 222244.png
2239,/home/g6422078/Data Preprocessing/Training Set...,1,IMG_170.jpg
2240,/home/g6422078/Data Preprocessing/Training Set...,4,IMG_873.jpg


In [9]:
# stratified split hanya dari nonaugmented
from sklearn.model_selection import train_test_split

# split train vs temp (val+test)
df_train, df_temp = train_test_split(
    df_nonaug,
    test_size=0.3,
    stratify=df_nonaug["Cluster"],
    random_state=42
)

# split temp jadi val dan test
df_val, df_test = train_test_split(
    df_temp,
    test_size=1/3,
    stratify=df_temp["Cluster"],
    random_state=42
)

In [10]:
# masukkan versi augmented ke masing-masing split
def add_augmentations(df_split, df_aug):
    image_ids = df_split["image_id"].unique()
    df_aug_subset = df_aug[df_aug["image_id"].isin(image_ids)]
    return pd.concat([df_split, df_aug_subset], ignore_index=True)

In [11]:
# final dataset
df_train_final = add_augmentations(df_train, df_aug)
df_val_final   = add_augmentations(df_val, df_aug)
df_test_final  = add_augmentations(df_test, df_aug)

# ambil kolom yang dibutuhkan 
df_train_final = df_train_final[["image_path", "Cluster"]]
df_val_final   = df_val_final[["image_path", "Cluster"]]
df_test_final  = df_test_final[["image_path", "Cluster"]]

# don't forget to shuffle
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)
df_val_final   = df_val_final.sample(frac=1, random_state=42).reset_index(drop=True)
df_test_final  = df_test_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [12]:
# save
df_train_final.to_csv(r"D:\UNI LIFE\TUGAS AKHIR\Data Splitting\New (after preprocessing)\df_train.csv", index=False)
df_val_final.to_csv(r"D:\UNI LIFE\TUGAS AKHIR\Data Splitting\New (after preprocessing)\df_val.csv", index=False)
df_test_final.to_csv(r"D:\UNI LIFE\TUGAS AKHIR\Data Splitting\New (after preprocessing)\df_test.csv", index=False)

In [13]:
df_train_final.value_counts('Cluster')

Cluster
4    1496
1    1392
3    1228
2    1216
0     944
Name: count, dtype: int64

In [14]:
df_val_final.value_counts('Cluster')

Cluster
4    428
1    400
2    348
3    348
0    268
Name: count, dtype: int64

In [15]:
df_test_final.value_counts('Cluster')

Cluster
4    216
1    200
3    176
2    172
0    136
Name: count, dtype: int64